# 09 — Demo dos 4 fluxos LangGraph

Demonstra os 4 workflows obrigatórios da Tech Challenge Fase 3:

| Fluxo | Módulo | Cenário testado |
|---|---|---|
| Triagem Ginecológica | `lib.workflows.triagem` | Sangramento intenso + dor pélvica → emergência |
| Detecção de Violência | `lib.workflows.violencia` | Sinais clínicos múltiplos → alta suspeita → SINAN |
| Obstétrico | `lib.workflows.obstetrico` | Gestante 32 semanas com cefaleia + escotomas → pré-eclâmpsia |
| Prevenção | `lib.workflows.prevencao` | Paciente com mamografia atrasada → propor agendamento |

## Pré-requisitos

1. ✅ `05_gerar_dados_mock.ipynb` rodado (`hospital.db` populado)
2. ✅ `06_indexar_protocolos.ipynb` rodado (Chroma indexado)
3. ✅ Opcionalmente: adapter LoRA do fine-tuning (`03_treinar_qlora.ipynb`). Sem ele, usa modelo base.
4. ✅ Pasta `lib/` em `/MyDrive/AssistenteHospitalar/lib/`.

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes \
    langchain langchain-community langchain-huggingface langgraph \
    chromadb sentence-transformers python-dotenv

: 

In [ ]:
import os, sys, json
from pathlib import Path
from google.colab import drive
from dotenv import load_dotenv
from huggingface_hub import login

drive.mount('/content/drive', force_remount=True)

DRIVE_BASE = '/content/drive/MyDrive/AssistenteHospitalar'
LIB_PATH   = DRIVE_BASE           # lib/ deve estar em /MyDrive/AssistenteHospitalar/lib/
DB_PATH    = f'{DRIVE_BASE}/files/hospital.db'
CHROMA_DIR = f'{DRIVE_BASE}/files/chroma'
COLLECTION = 'protocolos_saude_mulher'

sys.path.insert(0, LIB_PATH)
os.environ['HOSPITAL_DB_PATH'] = DB_PATH
os.environ['DRIVE_BASE']       = DRIVE_BASE

load_dotenv(f'{DRIVE_BASE}/.env')
login(token=os.getenv('HF_TOKEN'))
print('Setup ok.')

In [ ]:
# Carrega LLM (auto-detecta adapter LoRA da última run de treino, se houver)
# Defaults novos: max_new_tokens=256, repetition_penalty=1.2, no_repeat_ngram_size=4
from lib.llm import load_finetuned, build_chat_model
model, tokenizer = load_finetuned()
chat_model = build_chat_model(model, tokenizer)

# Carrega Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True},
)
vs = Chroma(collection_name=COLLECTION, embedding_function=embeddings, persist_directory=CHROMA_DIR)
retriever = vs.as_retriever(search_kwargs={'k': 4})

# Conecta DB
from lib import db
conn = db.connect()
print('Pronto:', vs._collection.count(), 'chunks no Chroma.')

In [ ]:
from lib.workflows import (
    build_triagem_workflow, build_violencia_workflow,
    build_obstetrico_workflow, build_prevencao_workflow,
)
wf_triagem    = build_triagem_workflow(chat_model, conn, retriever)
wf_violencia  = build_violencia_workflow(chat_model, conn, retriever)
wf_obstetrico = build_obstetrico_workflow(chat_model, conn, retriever)
wf_prevencao  = build_prevencao_workflow(chat_model, conn, retriever)
print('Workflows compilados.')

In [ ]:
# Pretty-printer para o estado final
def imprimir_resposta(titulo, estado):
    print('=' * 70)
    print(titulo)
    print('=' * 70)
    resp = estado.get('resposta_estruturada', estado)
    for k, v in resp.items():
        if k in ('raciocinio', 'fontes'):
            continue
        if isinstance(v, (list, dict)):
            print(f'\n## {k}:')
            print(json.dumps(v, indent=2, ensure_ascii=False, default=str))
        else:
            print(f'{k}: {v}')
    print('\n## raciocínio (trace dos nodes):')
    for r in resp.get('raciocinio', []):
        print(f'  • {r}')
    if resp.get('fontes'):
        print('\n## fontes consultadas:')
        for f in resp['fontes'][:5]:
            print(f'  • [{f.get("category")}] {f.get("doc_id")}')
    print()

## 1) Fluxo de Triagem Ginecológica

**Cenário:** paciente 32a com sangramento intenso há 3 dias, dor pélvica forte, atraso menstrual de 8 semanas.

Expectativa: o fluxo deve classificar como **emergência** (sangramento + dor + atraso = suspeita de gestação ectópica rota / aborto), pular a etapa de orientações ambulatoriais e encaminhar direto ao pronto-socorro.

In [ ]:
estado = wf_triagem.invoke({
    'queixa': ('Paciente 32 anos, sangramento intenso há 3 dias, '
               'dor pélvica forte irradiando para ombro, '
               'atraso menstrual de 8 semanas. Estável hemodinamicamente.'),
    'paciente_id': None,
})
imprimir_resposta('TRIAGEM GINECOLÓGICA — caso 1 (emergência)', estado)

In [ ]:
estado = wf_triagem.invoke({
    'queixa': ('Paciente 28 anos, corrimento amarelado com odor há 5 dias, '
               'sem febre, sem dor pélvica. Vida sexual ativa.'),
    'paciente_id': None,
})
imprimir_resposta('TRIAGEM GINECOLÓGICA — caso 2 (rotina)', estado)

## 2) Fluxo de Detecção de Violência Doméstica

**Cenário:** paciente comparece com múltiplos sinais clínicos sugestivos. Profissional descreve o caso.

Expectativa: extração de sinais → matriz de pontuação → **alta_suspeita** → protocolo de segurança ativado → equipe acionada → notificação SINAN registrada (se `confirmacao_clinica=True` e `paciente_id` informado).

In [ ]:
# Seleciona uma paciente real do mock (sem registro prévio para o teste ser limpo)
pid = conn.execute(
    'SELECT p.paciente_id FROM pacientes p '
    'LEFT JOIN registros_violencia rv ON rv.paciente_id = p.paciente_id '
    'WHERE rv.id IS NULL LIMIT 1'
).fetchone()['paciente_id']

estado = wf_violencia.invoke({
    'descricao_caso': ('Paciente 28a comparece com lesões equimóticas em locais não-expostos '
                       '(face medial das coxas, dorso), em múltiplas fases de cicatrização. '
                       'Acompanhante recusou deixar a paciente sozinha, respondendo por ela. '
                       'Histórico de 3 atendimentos prévios por queixas inespecíficas. '
                       'Relato de isolamento social progressivo nos últimos meses.'),
    'paciente_id': pid,
    'profissional': 'dr_ana_residente',
    'confirmacao_clinica': True,
})
imprimir_resposta(f'DETECÇÃO DE VIOLÊNCIA — paciente_id={pid}', estado)

In [ ]:
# Confirma que o log_acesso e o registro foram criados
logs = conn.execute(
    'SELECT timestamp, usuario, tabela, paciente_id, motivo '
    'FROM log_acesso WHERE paciente_id = ? ORDER BY id DESC LIMIT 3',
    (pid,),
).fetchall()
print('Logs de acesso (mais recentes):')
for l in logs:
    print(' ', dict(l))

reg = conn.execute(
    'SELECT id, tipo, data_atendimento, notificado_sinan FROM registros_violencia '
    'WHERE paciente_id = ? ORDER BY id DESC LIMIT 1', (pid,),
).fetchone()
print('\nRegistro criado:', dict(reg) if reg else 'nenhum')

In [ ]:
# Caso de baixo risco para contraste
estado = wf_violencia.invoke({
    'descricao_caso': ('Paciente 35a, queixa de cefaleia recorrente sem fator desencadeante claro, '
                       'baixa adesão a tratamentos prescritos.'),
    'paciente_id': None,
    'profissional': 'dr_ana_residente',
    'confirmacao_clinica': False,
})
imprimir_resposta('DETECÇÃO DE VIOLÊNCIA — caso 2 (sem alerta)', estado)

## 3) Fluxo Obstétrico

**Cenário:** gestante 32 semanas, IG = 32sem, cefaleia intensa + escotomas + edema súbito de face — suspeita de pré-eclâmpsia/eclâmpsia iminente.

Expectativa: avaliação de risco (deve identificar fatores ou classificar conforme sintomas) → detector de alarmes encontra cefaleia + visão turva → eh_emergencia=True → encaminhamento imediato ao PS obstétrico.

In [ ]:
estado = wf_obstetrico.invoke({
    'descricao_caso': ('Gestante 34a, G3P2A0, IG 32 semanas pela DUM. Quadro de cefaleia intensa '
                       'há 24h, escotomas, edema súbito de face, dor epigástrica em barra. '
                       'HAS gestacional diagnosticada na semana 28.'),
    'ig_semanas': 32,
    'paciente_id': None,
})
imprimir_resposta('OBSTÉTRICO — caso 1 (pré-eclâmpsia/HELLP)', estado)

In [ ]:
estado = wf_obstetrico.invoke({
    'descricao_caso': ('Primigesta 24a, IG 12 semanas pela DUM, sem antecedentes mórbidos. '
                       'Primeira consulta de pré-natal. Sem queixas atuais.'),
    'ig_semanas': 12,
    'paciente_id': None,
})
imprimir_resposta('OBSTÉTRICO — caso 2 (1ª consulta, risco habitual)', estado)

## 4) Fluxo de Prevenção

**Cenário:** paciente do mock no cenário `mamografia_atrasada` (idade 50-69 com mamografia >3a).

Expectativa: detectar exame atrasado → orientações preventivas → agendamento proposto → lembretes redigidos.

In [ ]:
# Encontra uma paciente com exame atrasado pra ter cenário rico
from lib import alertas
for r in conn.execute('SELECT paciente_id FROM pacientes ORDER BY paciente_id').fetchall():
    if alertas.exames_atrasados(conn, r['paciente_id']):
        pid_prev = r['paciente_id']
        break
else:
    pid_prev = 1
print(f'Paciente escolhida: {pid_prev}')

estado = wf_prevencao.invoke({'paciente_id': pid_prev})
imprimir_resposta(f'PREVENÇÃO — paciente_id={pid_prev}', estado)

## Diagramas dos 4 fluxos (Mermaid)

Gerados automaticamente pelo LangGraph. Cole no [Mermaid Live Editor](https://mermaid.live) ou no relatório técnico.

In [ ]:
for nome, wf in [('triagem', wf_triagem),
                 ('violencia', wf_violencia),
                 ('obstetrico', wf_obstetrico),
                 ('prevencao', wf_prevencao)]:
    print(f'\n--- {nome} ---')
    print(wf.get_graph().draw_mermaid())

In [ ]:
conn.close()
print('Demo concluído.')